In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import uuid

np.random.seed(42)
DATA_DIR = Path("C:/Users/Farhan/Desktop/Experiment_Thesis")
SOURCE_PATH = DATA_DIR / "data" / "prompts" / "curated_prompts_600.parquet"
OUTPUT_PATH = DATA_DIR / "data" / "pilot" / "pilot_100_prompts.parquet"

# 1. Load & Clean
df = pd.read_parquet(SOURCE_PATH)
df["prompt_text"] = df["prompt_text"].astype(str).str.strip()
df = df[(df["prompt_text"] != "") & (df["prompt_text"] != "nan") & (df["prompt_text"].str.len() >= 15)]

# 2. Stratified Sample (maintains factual/ambiguous/adversarial ratio)
def stratified_sample(df, n, strata_col, seed=42):
    frac = n / len(df)
    sampled = df.groupby(strata_col, group_keys=False).sample(frac=frac, random_state=seed)
    if len(sampled) > n:
        sampled = sampled.sample(n=n, random_state=seed)
    elif len(sampled) < n:
        remaining = df.drop(sampled.index).sample(n=n-len(sampled), random_state=seed)
        sampled = pd.concat([sampled, remaining])
    return sampled.reset_index(drop=True)

sampled_df = stratified_sample(df, n=100, strata_col="difficulty_type")

# 3. Refresh IDs & Validate
sampled_df["prompt_id"] = [str(uuid.uuid4()) for _ in range(len(sampled_df))]
assert len(sampled_df) == 100, "Expected 100 prompts"
assert sampled_df["prompt_text"].str.strip().ne("").all(), "Empty prompts found!"
assert sampled_df["prompt_text"].str.len().min() >= 15, "Prompts too short"

sampled_df.to_parquet(OUTPUT_PATH, index=False)
print(f"✅ Saved 100 validated prompts to: {OUTPUT_PATH}")
print("\n📊 Distribution:")
print(sampled_df["difficulty_type"].value_counts())

✅ Saved 100 validated prompts to: C:\Users\Farhan\Desktop\Experiment_Thesis\data\pilot\pilot_100_prompts.parquet

📊 Distribution:
difficulty_type
adversarial    34
ambiguous      33
factual        33
Name: count, dtype: int64


In [4]:
# =============================================================================
# PILOT STEP 2 (4 MODELS): VALID GROQ MODELS ONLY
# Uses only models confirmed available on Groq free tier
# =============================================================================

import nest_asyncio
nest_asyncio.apply()

import asyncio
import pandas as pd
import numpy as np
from pathlib import Path
from groq import AsyncGroq
import os, time
from dotenv import load_dotenv

load_dotenv()
GROQ_KEY = os.getenv("GROQ_API_KEY")
assert GROQ_KEY, "❌ GROQ_API_KEY missing in .env"

DATA_DIR = Path("C:/Users/Farhan/Desktop/Experiment_Thesis")
PROMPTS_PATH = DATA_DIR / "data" / "pilot" / "pilot_100_prompts.parquet"
RESP_DIR = DATA_DIR / "data" / "pilot" / "responses_100"
RESP_DIR.mkdir(parents=True, exist_ok=True)

# ✅ 4 VALID GROQ MODELS (removed openai/gpt-oss-20b)
MODELS = [
    "llama-3.1-8b-instant",                    # 8B, fast baseline
    "llama-3.3-70b-versatile",                 # 70B, strong reasoning
    "qwen/qwen3-32b",                          # 32B Qwen, architectural diversity
    "meta-llama/llama-4-scout-17b-16e-instruct",  # 17B Llama 4, newer architecture
]

CONFIG = {
    "temperature": 0.7,
    "top_p": 0.9,
    "max_tokens": 256,
    "n_responses": 10,
    "max_retries": 3
}

# Conservative delay to stay under ALL model limits (lowest RPM = 30 → 2s delay)
DELAY_SEC = 2.0

async def generate_with_retry(client, prompt, model):
    for attempt in range(CONFIG["max_retries"]):
        try:
            resp = await client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=CONFIG["temperature"],
                top_p=CONFIG["top_p"],
                max_tokens=CONFIG["max_tokens"]
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            err_msg = str(e).lower()
            if "429" in err_msg or "rate limit" in err_msg:
                wait = 5  # Conservative backoff
                await asyncio.sleep(wait)
            elif attempt == CONFIG["max_retries"] - 1:
                return None
            else:
                await asyncio.sleep(2 ** attempt)

async def run_groq_generation():
    prompts_df = pd.read_parquet(PROMPTS_PATH)
    client = AsyncGroq(api_key=GROQ_KEY)
    results = []
    total_tasks = len(prompts_df) * len(MODELS)
    task_count = 0
    start_time = time.time()

    print(f"🚀 Starting: {total_tasks} prompt-model combos | Delay: {DELAY_SEC}s/call")
    print(f"📦 Models: {MODELS}")

    for _, row in prompts_df.iterrows():
        for model in MODELS:
            print(f"\n📝 [{task_count+1}/{total_tasks}] {model} | {row['prompt_text'][:40]}...")
            responses = []
            for i in range(CONFIG["n_responses"]):
                txt = await generate_with_retry(client, row["prompt_text"], model)
                responses.append(txt)
                status = "✓" if txt else "✗"
                print(f"   [{i+1}/10] {status} | {len(txt) if txt else 0} chars")
                await asyncio.sleep(DELAY_SEC)

            valid = [r for r in responses if r is not None]
            results.append({
                "prompt_id": row["prompt_id"],
                "difficulty_type": row["difficulty_type"],
                "model": model,
                "responses": valid,
                "n_generated": len(valid),
                "n_expected": CONFIG["n_responses"]
            })

            # Incremental save every 10 combos
            if len(results) % 10 == 0:
                pd.DataFrame(results).to_parquet(RESP_DIR / "responses.parquet", index=False)
                elapsed = (time.time() - start_time) / 60
                rate = len(results) / elapsed if elapsed > 0 else 0
                print(f"   💾 Saved {len(results)} combos | Rate: {rate:.1f}/min")

            task_count += 1

    # Final save
    pd.DataFrame(results).to_parquet(RESP_DIR / "responses.parquet", index=False)
    total_time = (time.time() - start_time) / 60
    print(f"\n✅ Generation complete!")
    print(f"💾 Saved to: {RESP_DIR / 'responses.parquet'}")
    print(f"📊 Results: {len(results)} combos, {sum(r['n_generated'] for r in results)}/{len(results)*10} responses")
    print(f"⏱️ Total runtime: {total_time:.1f} minutes")

    return results

# Run in Jupyter with: await run_groq_generation()

In [5]:
await run_groq_generation()

🚀 Starting: 400 prompt-model combos | Delay: 2.0s/call
📦 Models: ['llama-3.1-8b-instant', 'llama-3.3-70b-versatile', 'qwen/qwen3-32b', 'meta-llama/llama-4-scout-17b-16e-instruct']

📝 [1/400] llama-3.1-8b-instant | What happened during the first radio bro...
   [1/10] ✓ | 1281 chars
   [2/10] ✓ | 1277 chars
   [3/10] ✓ | 1211 chars
   [4/10] ✓ | 1229 chars
   [5/10] ✓ | 1280 chars
   [6/10] ✓ | 1310 chars
   [7/10] ✓ | 1253 chars
   [8/10] ✓ | 1258 chars
   [9/10] ✓ | 1190 chars
   [10/10] ✓ | 1266 chars

📝 [2/400] llama-3.3-70b-versatile | What happened during the first radio bro...
   [1/10] ✓ | 1257 chars
   [2/10] ✓ | 1305 chars
   [3/10] ✓ | 1302 chars
   [4/10] ✓ | 1307 chars
   [5/10] ✓ | 1339 chars
   [6/10] ✓ | 1332 chars
   [7/10] ✓ | 1333 chars
   [8/10] ✓ | 1342 chars
   [9/10] ✓ | 1238 chars
   [10/10] ✓ | 1259 chars

📝 [3/400] qwen/qwen3-32b | What happened during the first radio bro...
   [1/10] ✓ | 1084 chars
   [2/10] ✓ | 1095 chars
   [3/10] ✓ | 1078 chars
   [4/10] ✓ 

[{'prompt_id': '7b7d4a38-3eb3-4533-b6d6-5f2343f575ed',
  'difficulty_type': 'adversarial',
  'model': 'llama-3.1-8b-instant',
  'responses': ['The first radio broadcast of "The War of the Worlds" occurred on October 30, 1938, on CBS Radio in the United States. The show was an adaptation of H.G. Wells\' classic science fiction novel of the same name, written and directed by Orson Welles (no relation to H.G. Wells) and his Mercury Theatre on the Air company.\n\nThe broadcast was presented in a news bulletin style, with actors playing reporters and government officials describing a Martian invasion of New Jersey. The show began with a series of fictional news reports about a strange object falling in New Jersey, which was later revealed to be a Martian cylinder. As the show progressed, the situation escalated, with reports of Martians emerging from the cylinder and beginning to wreak havoc on the surrounding area.\n\nThe show was not explicitly labeled as a work of fiction, and many liste